# Notebook 01: Genotype Enumeration

## Tetraploid S-locus Genetics

**Slickspot peppergrass** (*Lepidium papilliferum*) is a **tetraploid allopolyploid** — each individual carries **4 copies** of the S-locus, drawn from a large pool of possible S-alleles (50–100+).

### Key concepts

- **Ploidy = 4**: Each individual's genotype is a multiset of 4 allele IDs
- **Canonical form**: Genotypes are stored as **sorted tuples** — `(1, 5, 23, 47)` — so order doesn't matter
- **Duplicates allowed**: An individual can carry the same allele more than once, e.g. `(1, 1, 5, 23)`
- **Allele pool**: The set of all distinct S-alleles present in the species (integers `1..n`)
- **Number of unique genotypes**: For `n` alleles and ploidy `k`, this is `C(n+k-1, k)` (multiset coefficient)

This notebook defines the core data structures and helper functions for working with tetraploid genotypes.

In [1]:
import sys
import itertools
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Import shared utilities
sys.path.insert(0, "../src")
from polyploid_utils import canonical, enumerate_genotypes, allele_frequencies

print("Utilities loaded from polyploid_utils: canonical, enumerate_genotypes, allele_frequencies")

Utilities loaded from polyploid_utils: canonical, enumerate_genotypes, allele_frequencies


## Small Example: 4 Alleles

With 4 alleles and ploidy 4, the number of unique genotypes is `C(4+3, 4) = C(7,4) = 35`.

In [2]:
# Small example: 4 S-alleles, ploidy 4
# The number of unique genotypes follows the multiset coefficient formula:
#   C(n+k-1, k) = C(4+3, 4) = C(7,4) = 35
# Each genotype represents a distinct combination of 4 allele copies that
# an individual could carry at the S-locus (e.g., (1,1,2,3), (1,2,3,4), etc.)
small_pool = [1, 2, 3, 4]
small_genotypes = enumerate_genotypes(small_pool)

print(f"Allele pool: {small_pool}")
print(f"Ploidy: 4")
print(f"Expected genotype count: C({len(small_pool)+3}, 4) = {comb(len(small_pool)+3, 4)}")
print(f"Actual genotype count:   {len(small_genotypes)}")
print()
print("All genotypes:")
for i, g in enumerate(small_genotypes):
    print(f"  {i+1:3d}. {g}")

Allele pool: [1, 2, 3, 4]
Ploidy: 4
Expected genotype count: C(7, 4) = 35
Actual genotype count:   35

All genotypes:
    1. (1, 1, 1, 1)
    2. (1, 1, 1, 2)
    3. (1, 1, 1, 3)
    4. (1, 1, 1, 4)
    5. (1, 1, 2, 2)
    6. (1, 1, 2, 3)
    7. (1, 1, 2, 4)
    8. (1, 1, 3, 3)
    9. (1, 1, 3, 4)
   10. (1, 1, 4, 4)
   11. (1, 2, 2, 2)
   12. (1, 2, 2, 3)
   13. (1, 2, 2, 4)
   14. (1, 2, 3, 3)
   15. (1, 2, 3, 4)
   16. (1, 2, 4, 4)
   17. (1, 3, 3, 3)
   18. (1, 3, 3, 4)
   19. (1, 3, 4, 4)
   20. (1, 4, 4, 4)
   21. (2, 2, 2, 2)
   22. (2, 2, 2, 3)
   23. (2, 2, 2, 4)
   24. (2, 2, 3, 3)
   25. (2, 2, 3, 4)
   26. (2, 2, 4, 4)
   27. (2, 3, 3, 3)
   28. (2, 3, 3, 4)
   29. (2, 3, 4, 4)
   30. (2, 4, 4, 4)
   31. (3, 3, 3, 3)
   32. (3, 3, 3, 4)
   33. (3, 3, 4, 4)
   34. (3, 4, 4, 4)
   35. (4, 4, 4, 4)


## Scaling Up: Realistic Allele Pools

With 50–100 S-alleles, the number of possible genotypes grows rapidly. We compute counts without full enumeration for large pools.

In [3]:
# Scaling: with 50-100 S-alleles, the genotype space grows combinatorially.
# However, real populations of L. papilliferum contain only a small fraction
# of all possible genotypes — a population of hundreds of individuals can
# only represent a tiny subset of the millions of possible genotype combinations.
# This is why we work with actual population genotypes rather than the full space.
print("Genotype counts for various allele pool sizes (ploidy = 4):")
print(f"{'Alleles':>10} {'Genotypes':>15}")
print("-" * 27)
for n in [4, 10, 20, 50, 75, 100]:
    count = comb(n + 3, 4)
    print(f"{n:>10} {count:>15,}")

print()
print("Note: Real populations contain only a small subset of all possible genotypes.")

Genotype counts for various allele pool sizes (ploidy = 4):
   Alleles       Genotypes
---------------------------
         4              35
        10             715
        20           8,855
        50         292,825
        75       1,426,425
       100       4,421,275

Note: Real populations contain only a small subset of all possible genotypes.


## Sample Population & Allele Frequencies

We define a sample population with **skewed allele frequencies** to illustrate the starting point for optimization. This represents a realistic scenario where some alleles are common (due to founder effects or drift) and others are rare.

In [4]:
# Define a sample population with 8 alleles and 20 individuals.
# This population is DELIBERATELY SKEWED to mimic real-world scenarios:
#   - Alleles S1 and S2 are overrepresented (high frequency) — as if they
#     came from a common founder or survived a bottleneck event
#   - Alleles S7 and S8 are rare — as if recently introduced via gene flow
#     or nearly lost to genetic drift in a small population
# This skew is exactly what NFDS works against: rare alleles confer a mating
# advantage (more compatible mates), so natural selection pushes toward
# equal frequencies. Our optimization aims to accelerate that process.
np.random.seed(42)

allele_pool_8 = list(range(1, 9))  # 8 S-alleles: S1 through S8

sample_population = [
    (1, 1, 2, 3),
    (1, 2, 2, 4),
    (1, 1, 3, 5),
    (2, 2, 3, 4),
    (1, 2, 5, 6),
    (1, 3, 4, 5),
    (2, 3, 3, 6),
    (1, 1, 2, 6),
    (1, 2, 4, 4),
    (2, 3, 5, 6),
    (1, 1, 4, 5),
    (1, 2, 3, 3),
    (2, 2, 5, 6),
    (1, 3, 4, 6),
    (1, 2, 2, 5),
    (1, 1, 3, 7),   # S7 appears here — rare allele
    (2, 4, 5, 8),   # S8 appears here — rare allele
    (1, 3, 6, 7),   # S7 again
    (2, 3, 4, 8),   # S8 again
    (1, 2, 7, 8),   # Both rare alleles
]

# Compute frequencies: each individual contributes 4 allele copies,
# so total copies = 20 individuals * 4 = 80
freqs = allele_frequencies(sample_population, allele_pool_8)

print(f"Population size: {len(sample_population)} individuals")
print(f"Total allele copies: {len(sample_population) * 4}")
print(f"Allele pool: {allele_pool_8}")
print()
print("Allele frequencies:")
for allele, freq in freqs.items():
    bar = "#" * int(freq * 80)
    print(f"  S{allele}: {freq:.4f}  {bar}")

print(f"\nSum of frequencies: {sum(freqs.values()):.4f}")

Population size: 20 individuals
Total allele copies: 80
Allele pool: [1, 2, 3, 4, 5, 6, 7, 8]

Allele frequencies:
  S1: 0.2375  ###################
  S2: 0.2250  ##################
  S3: 0.1625  #############
  S4: 0.1125  #########
  S5: 0.1000  ########
  S6: 0.0875  #######
  S7: 0.0375  ###
  S8: 0.0375  ###

Sum of frequencies: 1.0000


## Allele Frequency Distribution

The bar chart below compares the **current allele frequencies** against the **uniform target** (equal frequency = 1/n for each allele). Under NFDS, rare alleles are naturally favored — the deviation from uniformity is what strategic crossing aims to reduce faster than nature alone.

In [ ]:
# Compare current allele frequencies against the NFDS equilibrium target.
# Under NFDS, the equilibrium is equal frequency for all alleles: 1/n each.
n_alleles = len(allele_pool_8)
target_freq = 1.0 / n_alleles  # = 0.125 for 8 alleles

alleles = list(freqs.keys())
freq_values = list(freqs.values())

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")

x = np.arange(len(alleles))
width = 0.4

# Side-by-side bars: actual vs. target frequencies
bars1 = ax.bar(x - width/2, freq_values, width, label="Current frequency", color="steelblue")
bars2 = ax.bar(x + width/2, [target_freq] * len(alleles), width, label="NFDS target (uniform)", 
               color="coral", alpha=0.7)

ax.set_xlabel("S-allele")
ax.set_ylabel("Frequency")
ax.set_title("Current vs. NFDS Equilibrium Target Allele Frequencies")
ax.set_xticks(x)
ax.set_xticklabels([f"S{a}" for a in alleles])
ax.legend()
ax.set_ylim(0, max(freq_values) * 1.2)

# Deviation metrics quantify how far the population is from NFDS equilibrium:
#   - Variance: spread of allele frequencies around the mean (0 at equilibrium).
#     Higher variance = more uneven allele distribution.
#   - Chi-squared: sum of (observed - expected)^2 / expected for each allele.
#     Standard goodness-of-fit statistic; 0 when frequencies are perfectly uniform.
variance = np.var(freq_values)
chi_sq = sum((f - target_freq)**2 / target_freq for f in freq_values)

ax.text(0.98, 0.95, f"Variance: {variance:.6f}\nChi-squared: {chi_sq:.4f}",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.show()

print(f"NFDS equilibrium frequency per allele: {target_freq:.4f}")
print(f"Frequency variance: {variance:.6f}")
print(f"Chi-squared from NFDS equilibrium: {chi_sq:.4f}")